# Model Training and Evaluation

Five classifiers are trained with the exact stratified split from Step 2. `Malware` is the positive class. The test set is used only for final evaluation.

In [1]:
from pathlib import Path
import json, time
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
dataset_path = root / 'dataset' / 'Obfuscated-MalMem2022.csv'
models_path, images_path, reports_path = root / 'models', root / 'images', root / 'reports'
images_path.mkdir(exist_ok=True); reports_path.mkdir(exist_ok=True)
RANDOM_STATE = 42; POSITIVE_CLASS = 'Malware'
sns.set_theme(style='whitegrid')

## Load data and exact Step 2 split

In [2]:
df = pd.read_csv(dataset_path)
config = json.loads((models_path / 'preprocessing_config.json').read_text())
features = config['feature_columns']
X = df[features].copy()
y = df[config['target_column']].map({'Benign': 0, 'Malware': 1}).astype('int8')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=config['test_size'], random_state=RANDOM_STATE, stratify=y)
print('Original shape:', df.shape)
print('Feature count:', len(features))
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)
print('Train distribution:', y_train.value_counts().to_dict())
print('Test distribution:', y_test.value_counts().to_dict())

Original shape: (58058, 57)
Feature count: 52
Train shape: (46446, 52) Test shape: (11612, 52)
Train distribution: {0: 23381, 1: 23065}
Test distribution: {0: 5846, 1: 5766}


## Define pipelines and train models

In [3]:
scaled = ColumnTransformer([('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), features)], remainder='drop')
unscaled = ColumnTransformer([('numeric', Pipeline([('imputer', SimpleImputer(strategy='median'))]), features)], remainder='drop')
models = {
 'Logistic Regression': Pipeline([('preprocess', scaled), ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))]),
 'Decision Tree': Pipeline([('preprocess', unscaled), ('model', DecisionTreeClassifier(random_state=RANDOM_STATE))]),
 'Random Forest': Pipeline([('preprocess', unscaled), ('model', RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1))]),
 'Support Vector Machine': Pipeline([('preprocess', scaled), ('model', SVC(random_state=RANDOM_STATE))]),
 'XGBoost': Pipeline([('preprocess', unscaled), ('model', XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.9, colsample_bytree=0.9, objective='binary:logistic', eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1, tree_method='hist'))])
}
results, predictions, matrices, trained = [], {}, {}, {}
for name, pipeline in models.items():
 start = time.perf_counter(); pipeline.fit(X_train, y_train); elapsed = time.perf_counter() - start
 pred = pipeline.predict(X_test)
 if pd.isna(pred).any(): raise ValueError(name + ' produced missing predictions')
 matrix = confusion_matrix(y_test, pred, labels=[0, 1])
 if matrix.shape != (2, 2): raise ValueError(name + ' confusion matrix has wrong shape')
 row = {'Model':name, 'Accuracy':accuracy_score(y_test,pred), 'Precision':precision_score(y_test,pred,pos_label=1,zero_division=0), 'Recall':recall_score(y_test,pred,pos_label=1,zero_division=0), 'F1-Score':f1_score(y_test,pred,pos_label=1,zero_division=0), 'Training Time (seconds)':elapsed}
 if not all(0 <= row[k] <= 1 for k in ['Accuracy','Precision','Recall','F1-Score']): raise ValueError(name + ' metric outside [0,1]')
 results.append(row); predictions[name]=pred; matrices[name]=matrix; trained[name]=pipeline
results_df = pd.DataFrame(results).sort_values(['F1-Score','Recall'], ascending=False).reset_index(drop=True)
results_df.to_csv(reports_path / 'model_results.csv', index=False)
print(results_df.to_string(index=False))

                 Model  Accuracy  Precision   Recall  F1-Score  Training Time (seconds)
         Decision Tree  1.000000   1.000000 1.000000  1.000000                 0.761824
         Random Forest  1.000000   1.000000 1.000000  1.000000                 1.580460
               XGBoost  1.000000   1.000000 1.000000  1.000000                 1.057923
Support Vector Machine  0.999656   0.999653 0.999653  0.999653                 2.052286
   Logistic Regression  0.999053   0.999306 0.998786  0.999046                 0.522182


## Visualizations, best model, and saved artifacts

In [4]:
for metric, filename, title in [('Accuracy','model_accuracy_comparison.png','Model Accuracy Comparison'),('Precision','model_precision_comparison.png','Model Precision Comparison'),('Recall','model_recall_comparison.png','Model Recall Comparison'),('F1-Score','model_f1_comparison.png','Model F1-Score Comparison')]:
 fig, ax = plt.subplots(figsize=(10,6)); ax.bar(results_df['Model'], results_df[metric]); ax.set_ylim(0,1.05); ax.set_title(title); ax.set_ylabel(metric); ax.tick_params(axis='x', rotation=25); fig.tight_layout(); fig.savefig(images_path / filename, dpi=300); plt.close(fig)
long = results_df.melt(id_vars='Model', value_vars=['Accuracy','Precision','Recall','F1-Score'], var_name='Metric', value_name='Score')
fig, ax = plt.subplots(figsize=(12,7)); sns.barplot(data=long, x='Model', y='Score', hue='Metric', ax=ax); ax.set_ylim(0,1); ax.tick_params(axis='x', rotation=25); fig.tight_layout(); fig.savefig(images_path / 'model_performance_comparison.png', dpi=300); plt.close(fig)
best = results_df.iloc[0]; best_name = best['Model']; best_matrix = matrices[best_name]
fig, ax = plt.subplots(figsize=(7,6)); sns.heatmap(best_matrix, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Benign','Malware'], yticklabels=['Benign','Malware'], ax=ax); ax.set_title('Best Model Confusion Matrix: ' + best_name); ax.set_xlabel('Predicted'); ax.set_ylabel('True'); fig.tight_layout(); fig.savefig(images_path / 'best_model_confusion_matrix.png', dpi=300); plt.close(fig)
metadata = {'model_name':best_name, 'selection_rule':'Highest F1-Score on this dataset and test split; Recall considered for ransomware detection.', 'positive_class':POSITIVE_CLASS, 'feature_count':len(features), 'feature_names':features, 'target_classes':['Benign', 'Malware'], 'random_state':RANDOM_STATE, 'evaluation_metrics':{k:float(best[k]) for k in ['Accuracy','Precision','Recall','F1-Score']}, 'confusion_matrix_labels':['Benign', 'Malware'], 'confusion_matrix':best_matrix.tolist(), 'preprocessing':'Median imputation in every pipeline; scaling only for Logistic Regression and SVM.'}
joblib.dump(trained[best_name], models_path / 'best_ransomware_model.joblib'); (models_path / 'best_model_metadata.json').write_text(json.dumps(metadata, indent=2))
report = 'Best-performing model on this dataset and test split\n' + best_name + '\n' + results_df.to_string(index=False) + '\nConfusion matrix: ' + str(best_matrix.tolist())
(reports_path / 'model_results.txt').write_text(report)
loaded = joblib.load(models_path / 'best_ransomware_model.joblib'); smoke = loaded.predict(X_test.head(5))
required = [reports_path/'model_results.csv', reports_path/'model_results.txt', models_path/'best_ransomware_model.joblib', models_path/'best_model_metadata.json'] + [images_path/n for n in ['model_accuracy_comparison.png','model_precision_comparison.png','model_recall_comparison.png','model_f1_comparison.png','model_performance_comparison.png','best_model_confusion_matrix.png']]
assert len(smoke)==5 and not pd.isna(smoke).any() and all(p.exists() and p.stat().st_size > 0 for p in required)
print('Best-performing model on this dataset and test split:', best_name); print('Recall:', best['Recall']); print('Confusion matrix:', best_matrix.tolist()); print('Reload smoke test:', smoke.tolist())

Best-performing model on this dataset and test split: Decision Tree
Recall: 1.0
Confusion matrix: [[5846, 0], [0, 5766]]
Reload smoke test: [1, 1, 0, 0, 1]
